In [1]:
import psycopg2
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sdadas/mmlw-retrieval-roberta-large')

db_params = {
    "host": "pgvector_snoql_lab", 
    "database": "vectordb",
    "user": "user",
    "password": "password",
    "port": 5432
}

conn = psycopg2.connect(**db_params)
cursor = conn.cursor()

/home/coder/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3262.58it/s]


In [9]:
print("--- Wyszukiwanie przelewów zawierających kontekst 'JEDZENIE' ---")
context_vector_1 = model.encode("jedzenie").tolist()

query_food = """
    SELECT sender, receiver, amount, title, (title_embedding <=> %s::vector) AS distance
    FROM transactions
    ORDER BY distance ASC
    LIMIT 50;
"""
cursor.execute(query_food, (context_vector_1,))
for row in cursor.fetchall():
    print(f"Od: {row[0]} | Do: {row[1]} | Kwota: {row[2]} PLN | Tytuł: {row[3]} (Dystans: {row[4]:.4f})")

--- Wyszukiwanie przelewów zawierających kontekst 'JEDZENIE' ---
Od: user18 | Do: user20 | Kwota: 215.00 PLN | Tytuł: jedzenie (Dystans: 0.0000)
Od: user9 | Do: user11 | Kwota: 125.00 PLN | Tytuł: jedzenie (Dystans: 0.0000)
Od: user18 | Do: user20 | Kwota: 215.00 PLN | Tytuł: jedzenie (Dystans: 0.0000)
Od: user9 | Do: user11 | Kwota: 125.00 PLN | Tytuł: jedzenie (Dystans: 0.0000)
Od: user18 | Do: user20 | Kwota: 215.00 PLN | Tytuł: jedzenie (Dystans: 0.0000)
Od: user9 | Do: user11 | Kwota: 125.00 PLN | Tytuł: jedzenie (Dystans: 0.0000)
Od: user14 | Do: user16 | Kwota: 175.00 PLN | Tytuł: jedzenie (Dystans: 0.0000)
Od: user14 | Do: user16 | Kwota: 175.00 PLN | Tytuł: jedzenie (Dystans: 0.0000)
Od: user18 | Do: user20 | Kwota: 215.00 PLN | Tytuł: jedzenie (Dystans: 0.0000)
Od: user9 | Do: user11 | Kwota: 125.00 PLN | Tytuł: jedzenie (Dystans: 0.0000)
Od: user14 | Do: user16 | Kwota: 175.00 PLN | Tytuł: jedzenie (Dystans: 0.0000)
Od: user14 | Do: user16 | Kwota: 175.00 PLN | Tytuł: jedzen

In [8]:
print("--- Wyszukiwanie przelewów zawierających kontekst 'ZWROT ŚRODKÓW' (Zakres 30 - 150 PLN) ---")
context_vector_2 = model.encode("zwrot środków").tolist()

query_refund = """
    SELECT sender, receiver, amount, title, (title_embedding <=> %s::vector) AS distance
    FROM transactions
    WHERE amount BETWEEN 30.00 AND 150.00
    ORDER BY distance ASC
    LIMIT 50;
"""
cursor.execute(query_refund, (context_vector_2,))
for row in cursor.fetchall():
    print(f"Od: {row[0]} | Do: {row[1]} | Kwota: {row[2]} PLN | Tytuł: {row[3]} (Dystans: {row[4]:.4f})")

--- Wyszukiwanie przelewów zawierających kontekst 'ZWROT ŚRODKÓW' (Zakres 30 - 150 PLN) ---
Od: user1 | Do: user2 | Kwota: 120.50 PLN | Tytuł: zwrot za obiad (Dystans: 0.0406)
Od: user1 | Do: user2 | Kwota: 120.50 PLN | Tytuł: zwrot za obiad (Dystans: 0.0406)
Od: user1 | Do: user2 | Kwota: 120.50 PLN | Tytuł: zwrot za obiad (Dystans: 0.0406)
Od: user1 | Do: user2 | Kwota: 120.50 PLN | Tytuł: zwrot za obiad (Dystans: 0.0406)
Od: user1 | Do: user2 | Kwota: 120.50 PLN | Tytuł: zwrot za obiad (Dystans: 0.0406)
Od: user11 | Do: user12 | Kwota: 35.00 PLN | Tytuł: rachunek (Dystans: 0.0868)
Od: user7 | Do: user9 | Kwota: 105.00 PLN | Tytuł: rachunek (Dystans: 0.0868)
Od: user17 | Do: user18 | Kwota: 90.00 PLN | Tytuł: rachunek (Dystans: 0.0868)
Od: user7 | Do: user9 | Kwota: 105.00 PLN | Tytuł: rachunek (Dystans: 0.0868)
Od: user11 | Do: user12 | Kwota: 35.00 PLN | Tytuł: rachunek (Dystans: 0.0868)
Od: user7 | Do: user9 | Kwota: 105.00 PLN | Tytuł: rachunek (Dystans: 0.0868)
Od: user11 | Do: 

In [8]:
cursor.close()
conn.close()
